In [1]:
import pandas as pd
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
ROOT = Path(".").resolve()
DATA_RAW = ROOT

In [3]:
import pandas as pd
df_debug = pd.read_csv("FEDFUNDS.csv")
print(df_debug.columns)

Index(['observation_date', 'FEDFUNDS'], dtype='object')


In [4]:
def load_fred_series(csv_name: str, value_col: str) -> pd.DataFrame:
    """https://fred.stlouisfed.org/series/FEDFUNDS"""
    path = DATA_RAW / csv_name
    df = pd.read_csv(path, na_values=".")
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df = df.rename(columns={"observation_date": "date"})

    df = df[["date", value_col]].dropna()

    return df

In [5]:
fed = load_fred_series("FEDFUNDS.csv", "FEDFUNDS")          
sp500 = load_fred_series("SP500.csv", "SP500")              

try:
    dfedtaru = load_fred_series("DFEDTARU.csv", "DFEDTARU")  
except FileNotFoundError:
    dfedtaru = None

try:
    dfedtarl = load_fred_series("DFEDTARL.csv", "DFEDTARL") 
except FileNotFoundError:
    dfedtarl = None

try:
    vix = load_fred_series("VIXCLS.csv", "VIXCLS")           
except FileNotFoundError:
    vix = None

In [6]:
fed.head(), sp500.head()


(        date  FEDFUNDS
 0 1954-07-01      0.80
 1 1954-08-01      1.22
 2 1954-09-01      1.07
 3 1954-10-01      0.85
 4 1954-11-01      0.83,
         date    SP500
 0 2020-11-16  3626.91
 1 2020-11-17  3609.53
 2 2020-11-18  3567.79
 3 2020-11-19  3581.87
 4 2020-11-20  3557.54)

In [7]:
fed = load_fred_series("FEDFUNDS.csv", "FEDFUNDS")      
sp500 = load_fred_series("SP500.csv", "SP500")         

def try_load_optional(csv_name, col_name):
    path = DATA_RAW / csv_name
    if path.exists():
        return load_fred_series(csv_name, col_name)
    else:
        print(f"Optional file {csv_name} not found, skipping.")
        return None

In [8]:
dfedtaru = try_load_optional("DFEDTARU.csv", "DFEDTARU")
dfedtarl = try_load_optional("DFEDTARL.csv", "DFEDTARL")
vix = try_load_optional("VIXCLS.csv", "VIXCLS")

print("\nPreview FEDFUNDS:")
print(fed.head())
print("\nPreview SP500:")
print(sp500.head())


Preview FEDFUNDS:
        date  FEDFUNDS
0 1954-07-01      0.80
1 1954-08-01      1.22
2 1954-09-01      1.07
3 1954-10-01      0.85
4 1954-11-01      0.83

Preview SP500:
        date    SP500
0 2020-11-16  3626.91
1 2020-11-17  3609.53
2 2020-11-18  3567.79
3 2020-11-19  3581.87
4 2020-11-20  3557.54


In [9]:
def to_monthly_mean(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    """Resample to monthly mean, return DataFrame indexed by month-end."""
    return (
        df.set_index("date")[value_col]
        .resample("M")
        .mean()
        .to_frame()
    )

def to_monthly_last(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    """Resample to monthly last observation, typical for price series."""
    return (
        df.set_index("date")[value_col]
        .resample("M")
        .last()
        .to_frame()
    )

fed_m = to_monthly_mean(fed, "FEDFUNDS").rename(columns={"FEDFUNDS": "rate_level"})

sp500_m = to_monthly_last(sp500, "SP500").rename(columns={"SP500": "sp500_close"})

In [10]:
if dfedtaru is not None:
    dfedtaru_m = to_monthly_mean(dfedtaru, "DFEDTARU")
if dfedtarl is not None:
    dfedtarl_m = to_monthly_mean(dfedtarl, "DFEDTARL")
if vix is not None:
    vix_m = to_monthly_mean(vix, "VIXCLS").rename(columns={"VIXCLS": "vix_mean"})

In [11]:
sp500_daily = sp500.set_index("date")["SP500"].sort_index()
sp500_daily_ret = sp500_daily.pct_change()

realized_vol_m = (
    sp500_daily_ret
    .resample("M")
    .std()
    .to_frame(name="realized_vol")
)

market_m = sp500_m.join(realized_vol_m, how="inner")

market_m["monthly_return"] = market_m["sp500_close"].pct_change()

for h in [1, 3, 6]:
    market_m[f"ret_{h}m_fwd"] = market_m["sp500_close"].shift(-h) / market_m["sp500_close"] - 1

print("\nPreview monthly market data:")
print(market_m.head())


Preview monthly market data:
            sp500_close  realized_vol  monthly_return  ret_1m_fwd  ret_3m_fwd  \
date                                                                            
2020-11-30      3621.63      0.008220             NaN    0.037121    0.052330   
2020-12-31      3756.07      0.005419        0.037121   -0.011137    0.057725   
2021-01-31      3714.24      0.010729       -0.011137    0.026091    0.125713   
2021-02-28      3811.15      0.009019        0.026091    0.042439    0.103108   
2021-03-31      3972.89      0.010431        0.042439    0.052425    0.081706   

            ret_6m_fwd  
date                    
2020-11-30    0.160834  
2020-12-31    0.144148  
2021-01-31    0.183354  
2021-02-28    0.186697  
2021-03-31    0.084233  


In [12]:
panel = fed_m.join(market_m, how="inner")

panel["rate_change"] = panel["rate_level"].diff()

if dfedtaru is not None:
    panel = panel.join(dfedtaru_m.rename(columns={"DFEDTARU": "target_upper"}), how="left")
if dfedtarl is not None:
    panel = panel.join(dfedtarl_m.rename(columns={"DFEDTARL": "target_lower"}), how="left")
if vix is not None:
    panel = panel.join(vix_m, how="left")

rate_median = panel["rate_level"].median()
panel["high_rate_regime"] = (panel["rate_level"] >= rate_median).astype(int)
panel["rising_rate_regime"] = (panel["rate_change"] > 0).astype(int)

panel = panel.dropna(subset=["monthly_return", "realized_vol"])

print("\nIntegrated panel preview:")
print(panel.head())
print("\nPanel length:", len(panel))



Integrated panel preview:
            rate_level  sp500_close  realized_vol  monthly_return  ret_1m_fwd  \
date                                                                            
2020-12-31        0.09      3756.07      0.005419        0.037121   -0.011137   
2021-01-31        0.09      3714.24      0.010729       -0.011137    0.026091   
2021-02-28        0.08      3811.15      0.009019        0.026091    0.042439   
2021-03-31        0.07      3972.89      0.010431        0.042439    0.052425   
2021-04-30        0.07      4181.17      0.006828        0.052425    0.005487   

            ret_3m_fwd  ret_6m_fwd  rate_change  target_upper  target_lower  \
date                                                                          
2020-12-31    0.057725    0.144148         0.00          0.25           0.0   
2021-01-31    0.125713    0.183354         0.00          0.25           0.0   
2021-02-28    0.103108    0.186697        -0.01          0.25           0.0   
2021-03-31

In [15]:
import os
import pandas as pd

RAW_DIR = "data/raw"

os.makedirs(RAW_DIR, exist_ok=True)

def main():
    """
    Placeholder script:
    - If you already downloaded FRED CSVs manually, this script documents
      where to put them and checks that they exist.
    - Expected files:
        data/raw/FEDFUNDS.csv
        data/raw/SP500.csv
    """
    fed_path = os.path.join(RAW_DIR, "FEDFUNDS.csv")
    sp_path = os.path.join(RAW_DIR, "SP500.csv")

    missing = []
    for p in [fed_path, sp_path]:
        if not os.path.exists(p):
            missing.append(p)

    if missing:
        print("The following files are missing. Please download from FRED and place them here:")
        for m in missing:
            print("  -", m)
    else:
        print("All expected raw files are present.")
        # Quick peek
        fed = pd.read_csv(fed_path).head()
        sp = pd.read_csv(sp_path).head()
        print("\nFEDFUNDS preview:\n", fed)
        print("\nSP500 preview:\n", sp)


if __name__ == "__main__":
    main()


The following files are missing. Please download from FRED and place them here:
  - data/raw/FEDFUNDS.csv
  - data/raw/SP500.csv


In [16]:
from pathlib import Path

ROOT = Path(".")

DATA_PROCESSED = ROOT / "data"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)


In [17]:
panel_path = DATA_PROCESSED / "monthly_panel.csv"
panel.to_csv(panel_path, index_label="month_end")
print("\nSaved monthly panel to:", panel_path)

core_cols = [
    "rate_level",
    "rate_change",
    "sp500_close",
    "monthly_return",
    "realized_vol",
    "ret_1m_fwd",
    "ret_3m_fwd",
    "ret_6m_fwd",
]

existing_core_cols = [c for c in core_cols if c in panel.columns]

summary_stats = panel[existing_core_cols].describe().T
corr_matrix = panel[existing_core_cols].corr()

summary_path = DATA_PROCESSED / "summary_stats.csv"
corr_path = DATA_PROCESSED / "correlations.csv"

summary_stats.to_csv(summary_path)
corr_matrix.to_csv(corr_path)

print("\n=== Summary statistics (core variables) ===")
print(summary_stats)
print("\n=== Correlation matrix (core variables) ===")
print(corr_matrix)

print("\nSaved summary stats to:", summary_path)
print("Saved correlations to:", corr_path)


Saved monthly panel to: data/monthly_panel.csv

=== Summary statistics (core variables) ===
                count         mean         std          min          25%  \
rate_level       59.0     3.156102    2.186540     0.060000     0.150000   
rate_change      59.0     0.067797    0.194724    -0.300000     0.000000   
sp500_close      59.0  4797.429831  860.841058  3585.620000  4132.040000   
monthly_return   59.0     0.011793    0.044007    -0.093396    -0.015979   
realized_vol     59.0     0.009730    0.004605     0.004042     0.006911   
ret_1m_fwd       59.0     0.010901    0.044018    -0.093396    -0.016613   
ret_3m_fwd       57.0     0.033475    0.066573    -0.164451    -0.012910   
ret_6m_fwd       54.0     0.062895    0.099211    -0.208544     0.004331   

                        50%          75%          max  
rate_level         4.330000     5.100000     5.330000  
rate_change        0.000000     0.100000     0.700000  
sp500_close     4522.680000  5545.680000  6840.200000 

In [19]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(".")
DATA_PROCESSED = ROOT / "data"
DOCS = ROOT / "docs"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

def load_fred_series_from_root(filename: str, value_col: str) -> pd.DataFrame:
    """
    Load a CSV sitting in the project root with columns:
    - observation_date
    - <value_col>
    """
    path = ROOT / filename
    df = pd.read_csv(path, na_values=".")
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df = df.rename(columns={"observation_date": "date"})
    df = df[["date", value_col]].dropna()
    return df


def try_load_optional_from_root(filename: str, value_col: str) -> pd.DataFrame | None:
    path = ROOT / filename
    if path.exists():
        return load_fred_series_from_root(filename, value_col)
    else:
        print(f"Optional file {filename} not found, skipping.")
        return None


def to_monthly_mean(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    return (
        df.set_index("date")[value_col]
        .resample("M")
        .mean()
        .to_frame()
    )


def to_monthly_last(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    return (
        df.set_index("date")[value_col]
        .resample("M")
        .last()
        .to_frame()
    )


fed = load_fred_series_from_root("FEDFUNDS.csv", "FEDFUNDS")
sp500 = load_fred_series_from_root("SP500.csv", "SP500")

dfedtaru = try_load_optional_from_root("DFEDTARU.csv", "DFEDTARU")
dfedtarl = try_load_optional_from_root("DFEDTARL.csv", "DFEDTARL")
vix = try_load_optional_from_root("VIXCLS.csv", "VIXCLS")

print("FEDFUNDS head:\n", fed.head())
print("\nSP500 head:\n", sp500.head())

fed_m = to_monthly_mean(fed, "FEDFUNDS").rename(columns={"FEDFUNDS": "rate_level"})
sp500_m = to_monthly_last(sp500, "SP500").rename(columns={"SP500": "sp500_close"})

dfedtaru_m = to_monthly_mean(dfedtaru, "DFEDTARU") if dfedtaru is not None else None
dfedtarl_m = to_monthly_mean(dfedtarl, "DFEDTARL") if dfedtarl is not None else None
vix_m = (
    to_monthly_mean(vix, "VIXCLS").rename(columns={"VIXCLS": "vix_mean"})
    if vix is not None else None
)

sp500_daily = sp500.set_index("date")["SP500"].sort_index()
sp500_daily_ret = sp500_daily.pct_change()

realized_vol_m = (
    sp500_daily_ret
    .resample("M")
    .std()
    .to_frame(name="realized_vol")
)

market_m = sp500_m.join(realized_vol_m, how="inner")
market_m["monthly_return"] = market_m["sp500_close"].pct_change()

for h in [1, 3, 6]:
    market_m[f"ret_{h}m_fwd"] = (
        market_m["sp500_close"].shift(-h) / market_m["sp500_close"] - 1
    )

print("\nMonthly market data head:\n", market_m.head())

panel = fed_m.join(market_m, how="inner")

panel["rate_change"] = panel["rate_level"].diff()

if dfedtaru_m is not None:
    panel = panel.join(
        dfedtaru_m.rename(columns={"DFEDTARU": "target_upper"}),
        how="left"
    )
if dfedtarl_m is not None:
    panel = panel.join(
        dfedtarl_m.rename(columns={"DFEDTARL": "target_lower"}),
        how="left"
    )
if vix_m is not None:
    panel = panel.join(vix_m, how="left")

rate_median = panel["rate_level"].median()
panel["high_rate_regime"] = (panel["rate_level"] >= rate_median).astype(int)
panel["rising_rate_regime"] = (panel["rate_change"] > 0).astype(int)

panel = panel.dropna(subset=["monthly_return", "realized_vol"])

print("\nIntegrated panel head:\n", panel.head())
print("Panel length:", len(panel))

# ----- save monthly_panel.csv -----
panel_path = DATA_PROCESSED / "monthly_panel.csv"
panel.to_csv(panel_path, index_label="month_end")
print("\nSaved monthly panel to:", panel_path)

# ----- summary stats + correlations -----
core_cols = [
    "rate_level",
    "rate_change",
    "sp500_close",
    "monthly_return",
    "realized_vol",
    "ret_1m_fwd",
    "ret_3m_fwd",
    "ret_6m_fwd",
]

existing_core_cols = [c for c in core_cols if c in panel.columns]

summary_stats = panel[existing_core_cols].describe().T
corr_matrix = panel[existing_core_cols].corr()

summary_path = DATA_PROCESSED / "summary_stats.csv"
corr_path = DATA_PROCESSED / "correlations.csv"

summary_stats.to_csv(summary_path)
corr_matrix.to_csv(corr_path)

print("\n=== Summary statistics (core variables) ===")
print(summary_stats)
print("\n=== Correlation matrix (core variables) ===")
print(corr_matrix)
print("\nSaved summary stats to:", summary_path)
print("Saved correlations to:", corr_path)

col_descriptions = {
    "rate_level": "Monthly average effective federal funds rate (percent).",
    "rate_change": "Month-over-month change in the effective federal funds rate (percentage points).",
    "sp500_close": "S&P 500 month-end closing level.",
    "monthly_return": "Simple monthly return of the S&P 500 based on month-end close.",
    "realized_vol": "Realized monthly volatility: standard deviation of daily S&P 500 returns within the month.",
    "ret_1m_fwd": "1-month forward S&P 500 return from this month’s close.",
    "ret_3m_fwd": "3-month forward S&P 500 return from this month’s close.",
    "ret_6m_fwd": "6-month forward S&P 500 return from this month’s close.",
    "target_upper": "Monthly average federal funds target upper bound (if available).",
    "target_lower": "Monthly average federal funds target lower bound (if available).",
    "vix_mean": "Monthly average of the CBOE Volatility Index VIX (if available).",
    "high_rate_regime": "Indicator = 1 if rate_level is above the sample median, 0 otherwise.",
    "rising_rate_regime": "Indicator = 1 if monthly rate_change > 0, 0 otherwise.",
}

lines = ["# Data Dictionary: monthly_panel.csv", ""]
lines.append("| Column | Description |")
lines.append("|--------|-------------|")

for col in panel.columns:
    desc = col_descriptions.get(col, "(description to be added)")
    lines.append(f"| `{col}` | {desc} |")

data_dict_path = DOCS / "data_dictionary.md"
with data_dict_path.open("w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("\nData dictionary written to:", data_dict_path)
print("Pipeline complete.")


FEDFUNDS head:
         date  FEDFUNDS
0 1954-07-01      0.80
1 1954-08-01      1.22
2 1954-09-01      1.07
3 1954-10-01      0.85
4 1954-11-01      0.83

SP500 head:
         date    SP500
0 2020-11-16  3626.91
1 2020-11-17  3609.53
2 2020-11-18  3567.79
3 2020-11-19  3581.87
4 2020-11-20  3557.54

Monthly market data head:
             sp500_close  realized_vol  monthly_return  ret_1m_fwd  ret_3m_fwd  \
date                                                                            
2020-11-30      3621.63      0.008220             NaN    0.037121    0.052330   
2020-12-31      3756.07      0.005419        0.037121   -0.011137    0.057725   
2021-01-31      3714.24      0.010729       -0.011137    0.026091    0.125713   
2021-02-28      3811.15      0.009019        0.026091    0.042439    0.103108   
2021-03-31      3972.89      0.010431        0.042439    0.052425    0.081706   

            ret_6m_fwd  
date                    
2020-11-30    0.160834  
2020-12-31    0.144148  
2021

In [20]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(".")
DATA_PROCESSED = ROOT / "data"
FIG_DIR = ROOT / "results" / "figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(DATA_PROCESSED / "monthly_panel.csv", parse_dates=["month_end"])
panel = panel.set_index("month_end")

fig, ax1 = plt.subplots()

ax1.plot(panel.index, panel["sp500_close"])
ax1.set_xlabel("Date")
ax1.set_ylabel("S&P 500 Index")

ax2 = ax1.twinx()
ax2.plot(panel.index, panel["rate_level"])
ax2.set_ylabel("Federal Funds Rate (%)")

fig.tight_layout()
fig.savefig(FIG_DIR / "timeseries_sp500_fedfunds.png")
plt.close(fig)

fig, ax = plt.subplots()
ax.scatter(panel["rate_level"], panel["sp500_close"])
ax.set_xlabel("Federal Funds Rate (%)")
ax.set_ylabel("S&P 500 Index Level")

fig.tight_layout()
fig.savefig(FIG_DIR / "scatter_sp500_vs_fedfunds.png")
plt.close(fig)

print("Saved figures to", FIG_DIR)


Saved figures to results/figures


In [23]:

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path(".").resolve()
DATA_PROCESSED = ROOT / "data"
DOCS = ROOT / "docs"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

def load_fred_series_from_root(filename: str, value_col: str) -> pd.DataFrame:
    path = ROOT / filename
    df = pd.read_csv(path, na_values=".")
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df = df.rename(columns={"observation_date": "date"})
    df = df[["date", value_col]].dropna()
    return df


def try_load_optional_from_root(filename: str, value_col: str) -> pd.DataFrame | None:
    path = ROOT / filename
    if path.exists():
        return load_fred_series_from_root(filename, value_col)
    else:
        print(f"Optional file {filename} not found, skipping.")
        return None


def to_monthly_mean(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    return (
        df.set_index("date")[value_col]
        .resample("M")
        .mean()
        .to_frame()
    )


def to_monthly_last(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    return (
        df.set_index("date")[value_col]
        .resample("M")
        .last()
        .to_frame()
    )


def main():
    fed = load_fred_series_from_root("FEDFUNDS.csv", "FEDFUNDS")
    sp500 = load_fred_series_from_root("SP500.csv", "SP500")

    dfedtaru = try_load_optional_from_root("DFEDTARU.csv", "DFEDTARU")
    dfedtarl = try_load_optional_from_root("DFEDTARL.csv", "DFEDTARL")
    vix = try_load_optional_from_root("VIXCLS.csv", "VIXCLS")

    print("FEDFUNDS head:\n", fed.head())
    print("\nSP500 head:\n", sp500.head())

    fed_m = to_monthly_mean(fed, "FEDFUNDS").rename(columns={"FEDFUNDS": "rate_level"})
    sp500_m = to_monthly_last(sp500, "SP500").rename(columns={"SP500": "sp500_close"})

    dfedtaru_m = to_monthly_mean(dfedtaru, "DFEDTARU") if dfedtaru is not None else None
    dfedtarl_m = to_monthly_mean(dfedtarl, "DFEDTARL") if dfedtarl is not None else None
    vix_m = (
        to_monthly_mean(vix, "VIXCLS").rename(columns={"VIXCLS": "vix_mean"})
        if vix is not None else None
    )

    sp500_daily = sp500.set_index("date")["SP500"].sort_index()
    sp500_daily_ret = sp500_daily.pct_change()

    realized_vol_m = (
        sp500_daily_ret
        .resample("M")
        .std()
        .to_frame(name="realized_vol")
    )

    market_m = sp500_m.join(realized_vol_m, how="inner")
    market_m["monthly_return"] = market_m["sp500_close"].pct_change()

    for h in [1, 3, 6]:
        market_m[f"ret_{h}m_fwd"] = (
            market_m["sp500_close"].shift(-h) / market_m["sp500_close"] - 1
        )

    print("\nMonthly market data head:\n", market_m.head())

    panel = fed_m.join(market_m, how="inner")
    panel["rate_change"] = panel["rate_level"].diff()

    if dfedtaru_m is not None:
        panel = panel.join(
            dfedtaru_m.rename(columns={"DFEDTARU": "target_upper"}),
            how="left"
        )
    if dfedtarl_m is not None:
        panel = panel.join(
            dfedtarl_m.rename(columns={"DFEDTARL": "target_lower"}),
            how="left"
        )
    if vix_m is not None:
        panel = panel.join(vix_m, how="left")

    rate_median = panel["rate_level"].median()
    panel["high_rate_regime"] = (panel["rate_level"] >= rate_median).astype(int)
    panel["rising_rate_regime"] = (panel["rate_change"] > 0).astype(int)

    panel = panel.dropna(subset=["monthly_return", "realized_vol"])

    print("\nIntegrated panel head:\n", panel.head())
    print("Panel length:", len(panel))

    panel_path = DATA_PROCESSED / "monthly_panel.csv"
    panel.to_csv(panel_path, index_label="month_end")
    print("\nSaved monthly panel to:", panel_path)

    core_cols = [
        "rate_level",
        "rate_change",
        "sp500_close",
        "monthly_return",
        "realized_vol",
        "ret_1m_fwd",
        "ret_3m_fwd",
        "ret_6m_fwd",
    ]
    existing_core_cols = [c for c in core_cols if c in panel.columns]

    summary_stats = panel[existing_core_cols].describe().T
    corr_matrix = panel[existing_core_cols].corr()

    summary_path = DATA_PROCESSED / "summary_stats.csv"
    corr_path = DATA_PROCESSED / "correlations.csv"

    summary_stats.to_csv(summary_path)
    corr_matrix.to_csv(corr_path)

    print("\n=== Summary statistics (core variables) ===")
    print(summary_stats)
    print("\n=== Correlation matrix (core variables) ===")
    print(corr_matrix)
    print("\nSaved summary stats to:", summary_path)
    print("Saved correlations to:", corr_path)

    col_descriptions = {
        "rate_level": "Monthly average effective federal funds rate (percent).",
        "rate_change": "Month-over-month change in the effective federal funds rate (percentage points).",
        "sp500_close": "S&P 500 month-end closing level.",
        "monthly_return": "Simple monthly return of the S&P 500 based on month-end close.",
        "realized_vol": "Realized monthly volatility: standard deviation of daily S&P 500 returns within the month.",
        "ret_1m_fwd": "1-month forward S&P 500 return from this month’s close.",
        "ret_3m_fwd": "3-month forward S&P 500 return from this month’s close.",
        "ret_6m_fwd": "6-month forward S&P 500 return from this month’s close.",
        "target_upper": "Monthly average federal funds target upper bound (if available).",
        "target_lower": "Monthly average federal funds target lower bound (if available).",
        "vix_mean": "Monthly average of the CBOE Volatility Index VIX (if available).",
        "high_rate_regime": "Indicator = 1 if rate_level is above the sample median, 0 otherwise.",
        "rising_rate_regime": "Indicator = 1 if monthly rate_change > 0, 0 otherwise.",
    }

    lines = ["# Data Dictionary: monthly_panel.csv", ""]
    lines.append("| Column | Description |")
    lines.append("|--------|-------------|")

    for col in panel.columns:
        desc = col_descriptions.get(col, "(description to be added)")
        lines.append(f"| `{col}` | {desc} |")

    data_dict_path = DOCS / "data_dictionary.md"
    with data_dict_path.open("w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("\nData dictionary written to:", data_dict_path)
    print("Pipeline complete.")


if __name__ == "__main__":
    main()


FEDFUNDS head:
         date  FEDFUNDS
0 1954-07-01      0.80
1 1954-08-01      1.22
2 1954-09-01      1.07
3 1954-10-01      0.85
4 1954-11-01      0.83

SP500 head:
         date    SP500
0 2020-11-16  3626.91
1 2020-11-17  3609.53
2 2020-11-18  3567.79
3 2020-11-19  3581.87
4 2020-11-20  3557.54

Monthly market data head:
             sp500_close  realized_vol  monthly_return  ret_1m_fwd  ret_3m_fwd  \
date                                                                            
2020-11-30      3621.63      0.008220             NaN    0.037121    0.052330   
2020-12-31      3756.07      0.005419        0.037121   -0.011137    0.057725   
2021-01-31      3714.24      0.010729       -0.011137    0.026091    0.125713   
2021-02-28      3811.15      0.009019        0.026091    0.042439    0.103108   
2021-03-31      3972.89      0.010431        0.042439    0.052425    0.081706   

            ret_6m_fwd  
date                    
2020-11-30    0.160834  
2020-12-31    0.144148  
2021

In [24]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(".").resolve()
DATA_PROCESSED = ROOT / "data"
FIG_DIR = ROOT / "results" / "figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)

def main():
    panel = pd.read_csv(DATA_PROCESSED / "monthly_panel.csv", parse_dates=["month_end"])
    panel = panel.set_index("month_end")

    fig, ax1 = plt.subplots()
    ax1.plot(panel.index, panel["sp500_close"])
    ax1.set_xlabel("Date")
    ax1.set_ylabel("S&P 500 Index")

    ax2 = ax1.twinx()
    ax2.plot(panel.index, panel["rate_level"])
    ax2.set_ylabel("Federal Funds Rate (%)")

    fig.tight_layout()
    fig.savefig(FIG_DIR / "timeseries_sp500_fedfunds.png")
    plt.close(fig)

    fig, ax = plt.subplots()
    ax.scatter(panel["rate_level"], panel["sp500_close"])
    ax.set_xlabel("Federal Funds Rate (%)")
    ax.set_ylabel("S&P 500 Index Level")

    fig.tight_layout()
    fig.savefig(FIG_DIR / "scatter_sp500_vs_fedfunds.png")
    plt.close(fig)

    print("Saved figures to", FIG_DIR)


if __name__ == "__main__":
    main()


Saved figures to /Users/amy/Desktop/Classes/IS477/results/figures
